# 24 · Reproject points (forward) against pyproj

**Question.** Does Fieldwork's `reproject` widget, which projects CRS84 longitude and
latitude into a WGS84 UTM zone with proj4, agree with PROJ through `pyproj`?

**Tolerance, stated before running: 1 mm.** Fieldwork stores projected coordinates to
three decimals, so rounding alone can displace a point by up to half a millimetre on each
axis — about 0.7 mm once combined. One millimetre is the smallest tolerance that rounding
does not already exhaust, and it is the figure [check 01](01-reproject-vs-pyproj.ipynb)
states for the opposite direction.

**Why this exists.** Check 01 recomputes the **inverse** transform — UTM to CRS84 — which
is what `reproject` did until release 0.1.0. From 0.2.0 the widget projects **forward**,
and that direction had no evidence: the widget catalog reported it as *"agrees · earlier
release"*, because the only check covering the widget covered a contract it no longer has.

**A round trip is not parity.** Fieldwork's forward and inverse transforms come from the
same proj4 build. A fault in proj4's UTM would round-trip to nine decimal places and still
be wrong in both directions. Only an independent implementation can say otherwise.

In [ ]:
# pyproj ships in the Pyodide distribution rather than as a wheel on this site, so it
# loads with pyodide_js.loadPackage. Outside Pyodide this cell does nothing.
import sys

if "pyodide" in sys.modules:
    import pyodide_js
    await pyodide_js.loadPackage("pyproj")

In [ ]:
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_24_project_forward import run, TOLERANCE_M

report = run(Path.cwd() / "fixtures" / "project-forward-utm30n-001.json")
print(json.dumps({k: report[k] for k in
                  ["environment", "fixture", "toleranceM", "points",
                   "maxSeparationM", "declaredCodeMatchesDefinition", "agrees"]}, indent=2))

The fixture's integrity is verified before anything is compared. A mismatch stops the
notebook: a result computed against an unidentified fixture is not evidence.

In [ ]:
for row in sorted(report["rows"], key=lambda r: -r["separationM"]):
    print(f"{row['id']:>4}  {row['input']['longitude']:>6.1f}, {row['input']['latitude']:>5.1f}"
          f"   reference {row['reference']['easting']:>12.3f}, {row['reference']['northing']:>12.3f}"
          f"   fieldwork {row['fieldwork']['easting']:>12.3f}, {row['fieldwork']['northing']:>12.3f}"
          f"   separation {row['separationM'] * 1000:6.3f} mm")

## Reading the residual

The separations are at the scale of the stored precision, not of the projection. Fieldwork
rounds eastings and northings to three decimals; the worst disagreement a correct
implementation can show under that rounding is

$$\\sqrt{(0.0005)^2 + (0.0005)^2} \\approx 0.00071\\,\\text{m}$$

so a residual at or below that bound says the two implementations agree and the file
cannot record the agreement more finely. A residual materially above it would be a real
disagreement rather than rounding.

The points are spread deliberately: on the central meridian at −3°, where the easting is
exactly 500,000 by construction, and at both edges of the six-degree band where the scale
factor is furthest from one. Latitudes run from 0.5° to 70°, because a metre of northing
is a very different number of degrees at each.

In [ ]:
# The bound the residual is read against, computed rather than asserted.
import math

half_unit = 0.5 * 10 ** -report["fixture"]["coordinateDecimals"]
bound = math.hypot(half_unit, half_unit)
print(f"rounding bound          {bound * 1000:.3f} mm")
print(f"worst separation        {report['maxSeparationM'] * 1000:.3f} mm")
print(f"within rounding alone   {report['maxSeparationM'] <= bound}")
print(f"within stated tolerance {report['maxSeparationM'] <= TOLERANCE_M}")

## The check that the coordinates alone cannot make

Fieldwork builds two things from the same zone number and hemisphere: the proj4 string it
projects with, and the `EPSG:326xx` label its receipts carry. They are constructed
separately, so they can disagree, and **no comparison of coordinates would notice**. Every
point would agree with the definition that was used, while the receipts named a different
frame to anyone reading them afterwards.

So this check recomputes from the declared *definition* and then asks PROJ whether the
declared *code* is the same frame. Measured by mutating the fixture on 2026-10-14:

| Fixture altered | Separation | Code matches | Verdict |
| --- | --- | --- | --- |
| nothing | 0.62 mm | yes | agrees |
| one easting moved 2 mm | 2.13 mm | yes | disagrees |
| definition's zone 30 → 31 | 671,519 m | no | disagrees |
| **code `EPSG:32630` → `EPSG:32631`** | **0.62 mm** | **no** | **disagrees** |
| definition made southern | 10,000,000 m | no | disagrees |

The fourth row is the one that matters. The coordinates are untouched and agree to the
same 0.62 mm; only the label is wrong. Without the frame comparison this check would have
passed a widget that misnames the projection in every receipt it writes.

## What this check establishes, and what it does not

It establishes that two independent implementations of the WGS84 UTM forward projection —
proj4 in the browser and PROJ through pyproj — agree over this fixture's spread of zone 30
north, to the precision at which Fieldwork stores the result, and that the EPSG code the
widget reports names the frame it actually projected into.

It does not establish that zone 30 north is the right zone for any given data. The same
longitude and latitude can be projected into any zone and will produce a valid-looking
easting and northing; [check 01's Dive Deeper section](01-reproject-vs-pyproj.ipynb)
measures what choosing a defensible but different zone costs.

It does not establish anything about datums. Both implementations are given WGS84, and a
file on another datum needs a shift Fieldwork does not implement and this check does not
exercise.

It does not establish that the widget is used correctly in a workflow. Projected points
are refused by every Fieldwork step that measures, which is a rule in the application and
not a property of this projection.